In [1]:
%pip install -q "scrapling[fetchers]" pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import time
from datetime import datetime
from urllib.parse import urljoin

import pandas as pd
from scrapling.fetchers import Fetcher

In [3]:
# Konfigurasi

MAX_ARTICLES_PER_SOURCE = 30
REQUEST_DELAY = 1.0

OUTPUT_FILE = "news.csv"


In [4]:
# Pagination handling
PAGINATION_SELECTORS = [
    "a[rel='next']",
    "a.next",
    "a:contains('Selanjutnya')",
    "a:contains('Berikutnya')",
]

In [5]:
# Keywords
CATEGORY = "otomotif"

KEYWORDS = {
    "otomotif": [
        "otomotif",
        "mobil",
        "motor",
        "kendaraan",
        "mobil listrik",
        "motor listrik",
        "toyota",
        "honda",
        "yamaha",
        "hyundai",
        "suzuki",
        "wuling",
        "byd",
        "daihatsu",
        "mitsubishi",
    ]
}


In [6]:
# konfigurasi sumber
SOURCES = {
    "inews": {
        "otomotif": "https://www.inews.id/otomotif"
    },

    # "kompas": {
    #     "otomotif": "https://otomotif.kompas.com/"
    # }
}

In [7]:
# Pagination handling
PAGINATION_SELECTORS = [
    "a[rel='next']",
    "a.next",
    "a:contains('Selanjutnya')",
    "a:contains('Berikutnya')",
]
MAX_PAGES_PER_ARTICLE = None   # set to int to limit pages per article

def fetch_all_pages(start_url, site):
    """Ambil semua halaman artikel yang dipaginasikan dan gabungkan isinya."""
    cur = start_url
    full = ""
    pages = 0
    while True:
        page = fetch(cur, site.get('mode', 'http'))
        if page.status != 200:
            break
        part = extract_content(page, site.get('content_selectors', []))
        if part:
            full += ("\n\n" if full else "") + part
        nxt = None
        for sel in PAGINATION_SELECTORS:
            nxt = page.css(f"{sel}::attr(href)").get()
            if nxt:
                break
        if not nxt:
            break
        cur = urljoin(cur, nxt)
        pages += 1
        if MAX_PAGES_PER_ARTICLE is not None and pages >= MAX_PAGES_PER_ARTICLE:
            break
        time.sleep(REQUEST_DELAY)
    return full

In [8]:
# utility

def match_keywords(title, content, category):
    """
    Mencari keyword yang muncul pada judul atau isi artikel.
    """

    text = f"{title} {content}".lower()

    matched = []

    for keyword in KEYWORDS.get(category, []):

        if keyword.lower() in text:
            matched.append(keyword)

    return matched

In [9]:
# kategori
def get_article_links(page, source):
    """
    Mengambil URL artikel dari halaman kategori.
    """

    links = []

    if source == "inews":

        # Semua link yang berada di halaman.
        raw_links = page.css("a::attr(href)").getall()

        for url in raw_links:

            if not url:
                continue

            if "/read/" in url:

                links.append(url)

            elif "/otomotif/" in url:

                links.append(url)


    elif source == "kompas":

        raw_links = page.css("a::attr(href)").getall()

        for url in raw_links:

            if not url:
                continue

            if "kompas.com" in url:

                if "/otomotif/" in url or "/bola/" in url:
                    links.append(url)

    return links


In [10]:
# exstrack artikel
def scrape_article(url, source, category):
    """
    Mengambil detail artikel.
    """

    print(f"  -> Scraping: {url}")

    try:

        page = Fetcher.get(
            url,
            stealthy_headers=True
        )

    except Exception as e:

        print(f"     ERROR request: {e}")

        return None

    # TITLE

    title = ""

    title_candidates = [
        "h1",
        "meta[property='og:title']::attr(content)",
        "title::text",
    ]

    for selector in title_candidates:

        try:

            result = page.css(selector).get()

            if result:

                # title = clean_text(
                #     result if isinstance(result, str)
                #     else result.text
                # )
                title = result if isinstance(result, str) else result.text

                if title:
                    break

        except Exception:
            continue

    # DATE

    date = ""

    date_candidates = [
        "time::attr(datetime)",
        "time::text",
        "meta[property='article:published_time']::attr(content)",
    ]

    for selector in date_candidates:

        try:

            result = page.css(selector).get()

            if result:

                # date = clean_text(
                #     result if isinstance(result, str)
                #     else result.text
                # )
                date = result if isinstance(result, str) else result.text

                if date:
                    break

        except Exception:
            continue

    # CONTENT

    paragraphs = []

    content_selectors = [
        "article p::text",
        ".article-content p::text",
        ".read__content p::text",
        ".detail__body-text p::text",
        ".content p::text",
    ]

    for selector in content_selectors:

        try:

            paragraphs = page.css(selector).getall()

            if paragraphs:
                break

        except Exception:
            continue

    content = " ".join(
        # clean_text(p)
        p for p in paragraphs
        if p
    )

    # KEYWORD MATCHING

    matched_keywords = match_keywords(
        title,
        content,
        category
    )

    if not matched_keywords:

        return None

    # RESULT

    return {
        "source": source,
        "category": category,
        "title": title,
        "url": url, # normalize_url(url),
        "date": date,
        "keywords": ", ".join(matched_keywords),
        "content": content,
        "scraped_at": datetime.now().isoformat(),
    }

In [11]:
# scrap kategori
def scrape_category(source, category, category_url):

    print()
    print("=" * 70)
    print(f"SOURCE   : {source}")
    print(f"CATEGORY : {category}")
    print(f"URL      : {category_url}")
    print("=" * 70)

    try:

        page = Fetcher.get(
            category_url,
            stealthy_headers=True
        )

    except Exception as e:

        print(f"ERROR membuka halaman kategori: {e}")

        return []

    links = get_article_links(
        page,
        source
    )

    # Remove duplicates
    links = list(dict.fromkeys(links))

    # Convert relative URLs
    links = [
        urljoin(category_url, url)
        for url in links
    ]

    print(f"Article links ditemukan: {len(links)}")

    results = []

    for index, url in enumerate(
        links[:MAX_ARTICLES_PER_SOURCE],
        start=1
    ):

        print(
            f"\n[{index}/{min(len(links), MAX_ARTICLES_PER_SOURCE)}]"
        )

        result = scrape_article(
            url=url,
            source=source,
            category=category
        )

        if result:

            results.append(result)

            print(
                f"     MATCH: {result['keywords']}"
            )

        else:

            print(
                "     Tidak cocok dengan keyword."
            )

        time.sleep(REQUEST_DELAY)

    return results


In [12]:
# main
def main():

    all_results = []

    for source, categories in SOURCES.items():

        for category, url in categories.items():

            results = scrape_category(
                source=source,
                category=category,
                category_url=url
            )

            all_results.extend(results)

    # DATAFRAME

    df = pd.DataFrame(all_results)

    if df.empty:

        print()
        print("Tidak ada artikel yang ditemukan.")

        return

    # REMOVE DUPLICATES

    df = df.drop_duplicates(
        subset=["url"]
    )

    # SORT

    df = df.sort_values(
        by=["source", "category"]
    )

    # SAVE CSV

    df.to_csv(
        OUTPUT_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    print()
    print("=" * 70)
    print("SCRAPING SELESAI")
    print("=" * 70)

    print(f"Total artikel : {len(df)}")
    print(f"Output        : {OUTPUT_FILE}")

    print()
    print(df[
        [
            "source",
            "category",
            "title",
            "date",
            "keywords",
        ]
    ].head(20))


if __name__ == "__main__":
    main()


SOURCE   : inews
CATEGORY : otomotif
URL      : https://www.inews.id/otomotif


[2026-10-01 10:07:36] INFO: Fetched (200) <GET https://www.inews.id/otomotif> (referer: https://www.google.com/)


Article links ditemukan: 33

[1/30]
  -> Scraping: https://www.inews.id/otomotif/motor


[2026-10-01 10:07:36] INFO: Fetched (200) <GET https://www.inews.id/otomotif/motor> (referer: https://www.google.com/)


     MATCH: motor

[2/30]
  -> Scraping: https://www.inews.id/otomotif/mobil


...


[28/30]
  -> Scraping: https://www.inews.id/otomotif/mobil/5-penyebab-bau-apek-di-dalam-mobil-ini-sering-tidak-disadari
     MATCH: mobil, kendaraan, suzuki


[2026-10-01 10:08:08] INFO: Fetched (200) <GET https://www.inews.id/otomotif/mobil/perkuat-pasar-toyota-bocorkan-akan-luncurkan-mobil-baru-di-akhir-tahun> (referer: https://www.google.com/)



[29/30]
  -> Scraping: https://www.inews.id/otomotif/mobil/perkuat-pasar-toyota-bocorkan-akan-luncurkan-mobil-baru-di-akhir-tahun
     MATCH: otomotif, mobil, motor, kendaraan, toyota


[2026-10-01 10:08:09] INFO: Fetched (200) <GET https://www.inews.id/otomotif/mobil/dashboard-mobil-bau-apek-ini-bagian-kabin-wajib-diperiksa> (referer: https://www.google.com/)



[30/30]
  -> Scraping: https://www.inews.id/otomotif/mobil/dashboard-mobil-bau-apek-ini-bagian-kabin-wajib-diperiksa
     MATCH: mobil

SCRAPING SELESAI
Total artikel : 29
Output        : news.csv

   source  category                                              title  \
0   inews  otomotif               Berita Motor Hari Ini - iNews Portal   
1   inews  otomotif        Berita Harian Otomotif Mobil - iNews Portal   
2   inews  otomotif  Berita Pelengkap Otomotif Mobil dan Motor - iN...   
3   inews  otomotif  Jual Beli Motor dan Mobil Baru dan Bekas - iNe...   
4   inews  otomotif  <h1 class="headerTitle">200 Mobil Modifikasi B...   
5   inews  otomotif  <h1 class="headerTitle">Uji Kekuatan, Tata Kub...   
6   inews  otomotif  <h1 class="headerTitle">Perkuat Pasar Otomotif...   
7   inews  otomotif  <h1 class="headerTitle">Dikepung Mobil China, ...   
8   inews  otomotif  <h1 class="headerTitle">Fernando Alonso Jadi P...   
9   inews  otomotif  <h1 class="headerTitle">Mobil Hybrid Isi